# 20.16 實際跑成熟模型，哪些結論才有證據？


模型對一個片段答「圓形」，還不能確認它有對齊聲音：它可能只猜最常見答案，或前處理根本沒有保留聲音事件。正式研究先固定一組畫面，將同一段聲音放在兩個合法時間，正解隨之改變；兩題都對才算這個配對成功。這承接[12.17](https://birdhackor.github.io/tiny-perceptron-vlm/12.17.html)的線索對照，也讓我們能分開「答案變了」與「真的變對」。

下列紀錄全部是手寫假設，尚不是Kev或Omni的輸出；本輪Kev的真實結果與Qwen2.5-Omni未取得評測結果的原因，另列在後面。先用小表看清計算方法，再讀實際紀錄。假設有三個情境家族，每家族兩個移時版本；每題候選依序為圓、方、以上皆不適合。raw logits是尚未除溫度的分數；T與接受門檻tau則假設早已分別由另一份calibration、policy資料選好，不能看這六題答案再改。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from math import exp

# 家族、版本、raw logits、正解index；以下都是人工假設值
rows = [
    ("p1", "早", [3.0, 0.0, -1.0], 0),
    ("p1", "晚", [0.0, 3.0, -1.0], 1),
    ("p2", "早", [1.2, 1.0, -1.0], 0),
    ("p2", "晚", [3.0, 0.0, -1.0], 1),
    ("p3", "早", [-1.0, 3.0, 0.0], 0),
    ("p3", "晚", [0.0, 3.0, -1.0], 1),
]
T, tau = 1.25, 0.85


def probabilities(raw_logits):
    scaled = [z / T for z in raw_logits]  # raw只縮放一次
    weights = [exp(z - max(scaled)) for z in scaled]
    return [w / sum(weights) for w in weights]


pairs = {}
accepted = []
correct = 0
for family, variant, logits, gold in rows:
    p = probabilities(logits)
    pred = max(range(len(p)), key=lambda i: p[i])
    ok = pred == gold
    correct += int(ok)
    if family not in pairs:
        pairs[family] = []
    pairs[family].append(ok)
    # 本例假設資訊足夠檢查皆為yes；none不自動接受
    if max(p) >= tau and pred != 2:
        accepted.append(ok)

paired = sum(len(v) == 2 and all(v) for v in pairs.values())
errors = sum(not ok for ok in accepted)
risk = errors / len(accepted) if accepted else None
print("逐題", correct, "/", len(rows))
print("兩題都對", paired, "/", len(pairs))
print("接受", len(accepted), "/", len(rows), "錯誤", errors)
print("coverage", round(len(accepted) / len(rows), 3))
print("accepted risk", None if risk is None else round(risk, 3))

`pairs`是普通字典；每遇到一個新家族，先用`if family not in pairs`建立空清單，再依序加入該家族各題的對錯。`all(v)`要求這個家族清單中的每題都對，前面的`len(v) == 2`另確認兩題都有紀錄。

印出逐題4/6、兩題都對1/3、接受5/6且其中錯2題，coverage為0.833，accepted risk為0.4。p2早的答案雖然對，分數未過門檻而轉交；p2晚卻以高分接受錯答。逐題正確率、配對正確率與接受後錯誤率因此回答不同問題。這張假設表的家族數為3；真資料仍須按共同底層情境定family，換ID不會製造獨立性，六個移時版本也不能算六個獨立家族。若沒有接受題，risk為None，表示無法由這批接受結果計算，而不是零風險。

這張表只示意指標的計算，不是兩種成熟模型共用的API。正式Kev只讀文字決策題；Omni影音案例則按固定規則解析文字答案，算配對正確率。介面沒有可靠候選raw logits時，不從生成文字捏造p_max或套T，兩種模型分開報告。

真實紀錄還必須能回答「模型究竟看到了什麼」。影音案例要保存解碼後的影格及時間、音訊採樣率與起點、processor真正取走的影格與時間座標，以及是否啟用影片聲軌等設定；只存原始影片檔名不足以排除抽樣漏掉線索。移時配對固定其他條件。完整驗證還應加入刪圖、刪聲音的不可判定控制；本輪自訓A有這兩種控制，成熟Omni的有限研究只預定24組影音移時配對，未包含刪線索測試，因此不能由它宣稱已通過缺少模態的拒答能力。processor改了可見材料時，要分開檢查前處理與模型回答，不能把所有錯都叫推理失敗。

Kev另保存adapter、base、head、tokenizer和程式pin，以及raw或上游已校準分數的身分。英語是4B卡片主要範圍，中文另報語言轉移，不混成一個平均。T只用校準資料，tau只用策略資料；final套固定規則。[9.18](https://birdhackor.github.io/tiny-perceptron-vlm/9.18.html)的coverage/risk與本例相同，而事後看正誤找最大coverage的曲線只能另作描述。

這次Kev真正看的材料沿用[19.23](https://birdhackor.github.io/tiny-perceptron-vlm/19.23.html)的三裝置規則世界：A的Choice動作、B的必要資訊、C的規準等級，以及另問A的Choice資訊。以下直接列出已封存final**第一個家族**的兩種state，讓上面的手寫圓／方表與真實研究分清楚：

```text
英文state：
A: task=save; permission=no; ready=no. B: task=play; permission=yes; ready=no. C: task=record; permission=no; ready=no.

中文state：
裝置A：任務save；許可否；就緒否。 裝置B：任務play；許可是；就緒否。 裝置C：任務record；許可否；就緒否。
```

只有可見欄位名與yes/no/unknown標記轉成中文；record/play/save等任務字仍照原紀錄。兩種state共用的Choice問題與候選描述仍是英文：

```text
Choose A's action. Require permission and ready known; no permission: stop; not ready: wait; else requested task.
save: Device A: save
wait: Device A: wait
record: Device A: record
play: Device A: play
stop: Device A: stop
none: None of these actions can be selected from the supplied facts and required-field contract.

```

依這題寫出的規則，兩個必要欄位都已知，先看permission=no，所以gold是stop，而非接著看ready=no就選wait。同一筆的B想play，只需permission已知，主info正解為yes；C想record，rubric數兩欄有幾個yes，故正解為0。另問A兩個欄位是否都已知的choice_info也是yes；資訊齊備不等於權限已獲准。本筆Kev在英文與中文state下的最高分候選都實際選wait，兩次都錯。這個例子用來對上材料、規則與回答，整體表現仍看同一批60家族的完整計數。

本輪Kev實驗使用固定Kev-4B／Qwen3.5-4B-Base，直接以FP32載入，head先設T=1取得raw logits，再另除一次校準溫度。80個英語calibration家族用Choice、資訊充分性、rubric三型等權的平均NLL選出T=1.8340；另一組60個policy家族選tau=0.8，在那組接受16題、錯0題。然後固定兩值，才量同一組60個final家族的兩種state寫法：

| state的寫法；問題與候選描述仍為英文 | Choice | 資訊充分性 | rubric |
| --- | --- | --- | --- |
| 英文state | 42／60 | 59／60 | 35／60 |
| 只把state可見欄位轉成中文 | 32／60 | 49／60 | 33／60 |

這是同60個情境的配對語言轉寫，三型問題也共享情境；不能把它加成120個獨立家族或兩份自然語料。三型表用各自預指定的代表題；接受規則另讀與Choice對應的充分性題，依它的yes預測及Choice的機率門檻決定接受。英語final按固定規則接受18／60題，其中錯1題，coverage為30%、接受風險約5.6%，超過原policy要求的5%。同一T／tau移到中文state只接受1／60題、錯0題；一題沒錯不足以建立穩健的低風險結論，也不能為提高coverage再看final挑門檻。

中文state的Choice平均NLL由raw的2.1189降為1.5782，但最高候選仍只有32／60正確。溫度改變報出的機率，沒有換掉最高分答案，和[9.18](https://birdhackor.github.io/tiny-perceptron-vlm/9.18.html)的機制一致。Kev與本課C的權重規模、預訓練、訓練資料及表示都不同，本輪只檢查各自條件下的結果，沒有隔離出架構本身的優劣。

另外八個dev家族的跨問題cache核對，完整重算與共享prefix的機率最大差為0.00000614，低於事前1e-5容差；Q1→Q2→Q1的重問差為0，KV、循環及卷積狀態的prefix內容指紋也都未變。這支持本次跨問題隔離的計算契約，語義正確率仍看上表；成熟Kev的選項重排沒有在本輪實測，不能沿用本課C的重排通過結果。固定版本、完整精度與原始檔指紋見[Kev實測摘要](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/omni-products/v1/kev-research-summary.json)。

Qwen2.5-Omni-3B的這次工作先完成24組合成英文語音／文字配對、24組影音移時配對與8個語音輸出提示，並核對輸入封包。影音是八張112×112的RGB影格、每秒兩張、共四秒，問題問發聲時的紅／藍色；它和自訓A的灰階圓／方形任務不同。語音輸入由FFmpeg的flite/slt合成，不能算自然人聲資料。

但雲端工作在等待執行容器就緒時，超過事先固定的八分鐘啟動上限，輸入還沒傳入研究worker；收尾紀錄確認該工作與Sandbox已停止。本輪沒有Qwen回答、實際processor輸出、生成WAV、正確率或延遲。因此這次失敗不能證明模型答不好、顯存不足，或學生電腦跑不動；公開實作所示的介面也不能冒充本次量測。確切版本、事前材料與停止證據見[Qwen執行摘要](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/omni-products/v1/qwen25-research-summary.json)。

聲音輸出則分三層：檔案完整可播放、說出的內容正確、聽起來自然。能讀出waveform只證明第一層；ASR轉回文字是可能出錯的內容代理，不能代替真人自然度。測量也要記介面究竟回傳整段聲音，還是逐次給可播放chunk。逐塊介面可另外量從請求到第一段可播放聲音的等待；整段介面只能等完整回傳才開始播放。本輪原訂使用後者，只能量整段生成完成時間；實際沒有取得聲音，便不填延遲，更不能把論文的234ms填成我們的實測。

最後，研究輸入、原始輸出、設定、失敗與版本都保留可追溯紀錄。3B只依其研究授權做本輪評估，不隨教材附權重，不用其輸出訓練A/C；Qwen3本輪只讀公開資料。只有來源、語言、前處理及介面條件都說清楚，少量案例才有可解讀的結果。

練習僅把tau改成0.95，先預測本例會沒有接受題、risk為None，再核對。這是在示意門檻的影響；正式final不能為了得到漂亮數字才這樣重挑。

原始依據：[Kev metrics固定碼版](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/kev/metrics.py)的coverage_at_error與分組重採樣；[Qwen2.5報告v1](https://arxiv.org/html/2503.20215v1)§2.2／2.4；[Qwen3報告v1](https://arxiv.org/html/2509.17765v1)§2.5；[逐資產授權紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-revision-20261029-omni/licenses.json)。
